#### setup

In [ ]:
import json
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader

from library.data_utils import *
from library.models import *
from library.training import *
from library.evaluation import *

In [ ]:
# device and seed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(seed=0)

In [ ]:
# parameters
dataset = 'synsum'
train_frac = 0.8  # a training fraction of 0.8 corresponds to 8000 training samples

#### data

In [ ]:
# load data and split
df = pd.read_csv(f"./data/datasets/{dataset}.csv", index_col=0)
train_df, val_df, test_df = make_splits(df=df, train_frac=train_frac, seed=0)

In [ ]:
# load embeddings
embeddings = torch.load(f"./data/embeddings/{dataset}.pt", weights_only=True)

In [ ]:
# set confounders
confounders_text = ['dysp', 'cough', 'pain', 'nasal', 'fever_none', 'fever_low', 'fever_high']
confounders_tabular = ['self_empl', 'asthma', 'smoking', 'COPD', 'winter','hay_fever']
confounders_full = confounders_text + confounders_tabular

#### set pseudo outcomes

In [ ]:
# load models
checkpoint_dir = './data/checkpoints/'
input_dim = len(confounders_full)

# e(x)
propensity_model = ClassificationHead(input_dim=input_dim, hidden_dim=64).to(device)
propensity_model.load_state_dict(torch.load(checkpoint_dir + 'propensity_model.pt', weights_only=True))

# mu0(x)
response_model_control = RegressionHead(input_dim=input_dim, hidden_dim=64).to(device)
response_model_control.load_state_dict(torch.load(checkpoint_dir + 'response_model_control.pt', weights_only=True))

# mu1(x)
response_model_treated = RegressionHead(input_dim=input_dim, hidden_dim=64).to(device)
response_model_treated.load_state_dict(torch.load(checkpoint_dir + 'response_model_treat.pt', weights_only=True))

In [ ]:
# set scores
train_df = compute_dr_scores(train_df, confounders_full, propensity_model, response_model_control, response_model_treated, device)
val_df = compute_dr_scores(val_df, confounders_full, propensity_model, response_model_control, response_model_treated, device)

#### train coarsened effect model

In [ ]:
# set parameters
input_dim = len(confounders_tabular) + 768
params = dict(
    hidden_dim=64,
    learning_rate=5e-4,
    weight_decay=0,
    batch_size=128,
    max_epochs=50,
    patience=5)

In [ ]:
# init data loaders
train_loader, val_loader = make_cate_loaders(train_df, val_df, confounders_tabular, confounders_full, embeddings, "DR", params['batch_size'])

# train cate model
aace_model = RegressionHead(input_dim=input_dim, hidden_dim=params["hidden_dim"]).to(device)
aace_model, info = train_cate(aace_model, train_loader, val_loader,device, lr=params['learning_rate'], weight_decay=params['weight_decay'],
                                max_epochs=params['max_epochs'], patience=params['patience'], seed=0, tabular=False)

# store
torch.save(aace_model.state_dict(), './data/checkpoints/aace_model.pt')

#### evaluation

In [ ]:
# set model to eval
aace_model.eval()

# init collectors
estimates, cates, M0s, M1s = [], [], [], []

# init test loader
test_loader = DataLoader(EvalDataset(test_df, confounders_tabular, confounders_full, embeddings), batch_size=1024, shuffle=False)

In [ ]:
# loop over test loader
with torch.inference_mode():
    for phi, x, cate, M0, M1 in test_loader:

        # forward pass
        phi = phi.to(device)
        effects = aace_model(phi).squeeze(-1)

        # collect
        estimates.append(effects)
        cates.append(cate.squeeze(-1))
        M0s.append(M0.squeeze(-1))
        M1s.append(M1.squeeze(-1))

# store
to_np = lambda parts: torch.cat(parts, dim=0).detach().cpu().numpy()
df_eval = pd.DataFrame({"est": to_np(estimates), "cate": to_np(cates), "M0": to_np(M0s), "M1": to_np(M1s)})

In [ ]:
# sort and eval
ranked = df_eval.sort_values('est', ascending=True).copy()
print(f"PEHE          : {pehe(ranked):.6f}")
print(f"Policy value  : {policy_value(ranked):.6f}")
print(f"AUTOC         : {autoc(ranked):.6f}")